# 02 — Sentiment Analysis & Comparison
Loads the raw tweets from `data/raw/`, runs `twitter-roberta-base-sentiment` on each one,
saves scored CSVs to `data/scored/`, then produces comparison plots across the Big 6.

In [ ]:
# Install dependencies if needed
# !pip install transformers torch scipy pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm.auto import tqdm

RAW_DIR    = Path("data/raw")
SCORED_DIR = Path("data/scored")
SCORED_DIR.mkdir(parents=True, exist_ok=True)

# Team colours for plots
TEAM_COLOURS = {
    "Arsenal":    "#EF0107",
    "Chelsea":    "#034694",
    "Liverpool":  "#C8102E",
    "Man City":   "#6CABDD",
    "Man United": "#DA020E",
    "Tottenham":  "#132257",
}

TEAMS = list(TEAM_COLOURS.keys())

## 1. Load model

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

print("Loading tokeniser and model (downloads on first run) ...")
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)
print(f"Using device: {DEVICE}")

# Label order for this model
LABELS = ["negative", "neutral", "positive"]

## 2. Scoring helper

In [ ]:
def preprocess(text: str) -> str:
    """Replace mentions and URLs with placeholders (model was trained this way)."""
    tokens = []
    for t in text.split():
        if t.startswith("@"):
            tokens.append("@user")
        elif t.startswith("http"):
            tokens.append("http")
        else:
            tokens.append(t)
    return " ".join(tokens)


def score_batch(texts: list[str], batch_size: int = 32) -> pd.DataFrame:
    """Score a list of texts; returns DataFrame with columns: negative, neutral, positive, label."""
    all_scores = []
    for i in range(0, len(texts), batch_size):
        batch = [preprocess(t) for t in texts[i : i + batch_size]]
        enc = tokeniser(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits.cpu().numpy()
        probs = softmax(logits, axis=1)
        all_scores.extend(probs.tolist())

    scores_df = pd.DataFrame(all_scores, columns=LABELS)
    scores_df["label"] = scores_df[LABELS].idxmax(axis=1)
    return scores_df

## 3. Score all tweets

In [ ]:
scored_frames = []

for team in TEAMS:
    csv_path = RAW_DIR / f"{team.lower().replace(' ', '_')}.csv"
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found — skipping {team}")
        continue

    df = pd.read_csv(csv_path)
    print(f"Scoring {team} ({len(df)} tweets) ...")

    scores = score_batch(df["text"].fillna("").tolist())
    df = pd.concat([df.reset_index(drop=True), scores], axis=1)

    out = SCORED_DIR / f"{team.lower().replace(' ', '_')}.csv"
    df.to_csv(out, index=False)
    scored_frames.append(df)
    print(f"  Saved → {out}")

df_all = pd.concat(scored_frames, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"], utc=True)
print(f"\n✓ Scoring complete — {len(df_all)} tweets scored")

## 4. Summary statistics

In [ ]:
summary = (
    df_all.groupby("team")[["negative", "neutral", "positive"]]
    .mean()
    .round(4)
    .sort_values("positive", ascending=False)
)
print("Average sentiment probabilities per team:\n")
print(summary.to_string())

## 5. Plots

In [ ]:
# ── Plot 1: Stacked bar — average sentiment breakdown per team ──────────────
fig, ax = plt.subplots(figsize=(10, 5))

bar_data = summary[["negative", "neutral", "positive"]]
bottom   = np.zeros(len(bar_data))
colours  = {"negative": "#d62728", "neutral": "#aec7e8", "positive": "#2ca02c"}

for col in ["negative", "neutral", "positive"]:
    ax.bar(bar_data.index, bar_data[col], bottom=bottom,
           label=col.capitalize(), color=colours[col], alpha=0.85)
    bottom += bar_data[col].values

ax.set_title("Average Sentiment Breakdown — Big 6 Premier League Teams", fontsize=13, pad=12)
ax.set_ylabel("Mean Probability")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(loc="upper right")
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("data/plot_stacked_bar.png", dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Violin — distribution of positive score per team ───────────────
fig, ax = plt.subplots(figsize=(11, 5))

team_order = summary.sort_values("positive", ascending=False).index.tolist()
palette    = {t: TEAM_COLOURS[t] for t in team_order}

sns.violinplot(
    data=df_all,
    x="team", y="positive",
    order=team_order,
    palette=palette,
    inner="quartile",
    ax=ax,
)

ax.set_title("Distribution of Positive Sentiment Score — Big 6", fontsize=13, pad=12)
ax.set_ylabel("Positive probability")
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("data/plot_violin_positive.png", dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Label distribution — % of tweets per sentiment class ───────────
label_counts = (
    df_all.groupby(["team", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["negative", "neutral", "positive"], fill_value=0)
)
label_pct = label_counts.div(label_counts.sum(axis=1), axis=0)
label_pct = label_pct.loc[team_order]   # same order as violin

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(label_pct))
for col in ["negative", "neutral", "positive"]:
    ax.bar(label_pct.index, label_pct[col], bottom=bottom,
           label=col.capitalize(), color=colours[col], alpha=0.85)
    bottom += label_pct[col].values

ax.set_title("Tweet Sentiment Label Distribution — Big 6", fontsize=13, pad=12)
ax.set_ylabel("Proportion of tweets")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("data/plot_label_distribution.png", dpi=150)
plt.show()

In [ ]:
# ── Plot 4: Sentiment over time (7-day rolling average, positive score) ────
df_all["date_only"] = df_all["date"].dt.date

fig, ax = plt.subplots(figsize=(13, 5))

for team in team_order:
    team_df = df_all[df_all["team"] == team].copy()
    daily   = team_df.groupby("date_only")["positive"].mean()
    daily.index = pd.to_datetime(daily.index)
    rolling = daily.rolling(7, min_periods=1).mean()
    ax.plot(rolling.index, rolling.values,
            label=team, color=TEAM_COLOURS[team], linewidth=2)

ax.set_title("7-Day Rolling Average Positive Sentiment (last 30 days)", fontsize=13, pad=12)
ax.set_ylabel("Positive probability (rolling avg)")
ax.set_xlabel("Date")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("data/plot_timeline.png", dpi=150)
plt.show()

In [ ]:
# ── Final ranked summary ────────────────────────────────────────────────────
print("\n=== FINAL RANKING (by average positive sentiment) ===\n")
ranked = summary.sort_values("positive", ascending=False)
for i, (team, row) in enumerate(ranked.iterrows(), 1):
    print(f"  {i}. {team:<14}  positive={row['positive']:.3f}  "
          f"neutral={row['neutral']:.3f}  negative={row['negative']:.3f}")